# ML — Employee Attrition: Preprocessing

**Goal:** Prepare the IBM HR dataset for machine learning — encode categoricals, scale numerics, handle class imbalance, and split into train/test sets.

---
**Sections:**
1. Load & Clean
2. Encode Categorical Variables
3. Feature Scaling
4. Train / Test Split
5. Handle Class Imbalance
6. Save Processed Data

## 1. Load & Clean

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

DATA_PATH = '../01_eda/data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Drop columns with zero variance — they carry no information
constant_cols = ['EmployeeCount', 'StandardHours', 'Over18']
df.drop(columns=constant_cols, inplace=True)

# Drop employee ID — not a predictive feature
df.drop(columns=['EmployeeNumber'], inplace=True)

print(f'Shape after cleanup: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 2. Encode Categorical Variables

In [ ]:
# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')

In [ ]:
# Encode target: Attrition Yes=1, No=0
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

# Binary columns — map directly
df['OverTime'] = (df['OverTime'] == 'Yes').astype(int)

# Ordinal column — BusinessTravel has a natural order
travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
df['BusinessTravel'] = df['BusinessTravel'].map(travel_map)

# Remaining categoricals — one-hot encoding
remaining_cats = ['Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']
df = pd.get_dummies(df, columns=remaining_cats, drop_first=True)

print(f'Shape after encoding: {df.shape}')
df.head()

## 3. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Attrition'])
y = df['Attrition']

# Stratify to preserve class balance in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]} samples')
print(f'Test size:  {X_test.shape[0]} samples')
print(f'\nAttrition rate in train: {y_train.mean():.1%}')
print(f'Attrition rate in test:  {y_test.mean():.1%}')

## 4. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale only numeric columns (not the binary/encoded ones)
num_cols = ['Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 'MonthlyIncome',
            'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
            'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany',
            'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

scaler = StandardScaler()

# Fit on train only — never on test (avoids data leakage)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Scaling applied.')
X_train.describe().round(2)

## 5. Class Imbalance

The dataset is imbalanced (~84% stayed, ~16% left). We handle this using `class_weight='balanced'` in the models rather than oversampling, to avoid artificially inflating the minority class.

In [ ]:
class_counts = y_train.value_counts()
print(f'Class distribution in training set:')
print(f'  Stayed (0): {class_counts[0]} ({class_counts[0]/len(y_train):.1%})')
print(f'  Left   (1): {class_counts[1]} ({class_counts[1]/len(y_train):.1%})')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Stayed (0)', 'Left (1)'], class_counts.values, color=['#4C72B0', '#DD8452'])
ax.set_title('Class Distribution — Training Set')
ax.set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save Processed Data

In [ ]:
import os

os.makedirs('data', exist_ok=True)

X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

# Save feature names for use in modeling notebook
pd.Series(X_train.columns.tolist()).to_csv('data/feature_names.csv', index=False)

print('Saved:')
print('  data/X_train.csv')
print('  data/X_test.csv')
print('  data/y_train.csv')
print('  data/y_test.csv')
print('  data/feature_names.csv')
print(f'\nFeatures: {X_train.shape[1]}')